In [1]:
import pandas as pd
import os
import json
import numpy as np
from os.path import dirname
from utils import get_case_ids

root_path = dirname(os.getcwd())

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/comuzzi/_processed/"
data_dir_graphs = root_path + "/data/datasets/comuzzi/graphs_repair/"

print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

/home/danbi/Projects/SANAGRAPH
/home/danbi/Projects/SANAGRAPH/data/datasets/original/
/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/_processed/
/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/


In [2]:
ACT_TIME_ONLY = True 

In [3]:
#ACT_TIME_ONLY = False

In [4]:
with open("dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [5]:
list(datasets_info.keys())

['BPI_Challenge_2013_open_problems',
 'sp2020',
 'Helpdesk',
 'BPI20_RequestForPayment',
 'BPI Challenge 2017 - Offer log',
 'BPI_Challenge_2012_W_Complete',
 'BPI_Challenge_2012_A',
 'bpi_2012_CZ',
 'bpi_2013_CZ',
 'large_log_CZ',
 'small_log_CZ',
 'sp2020_CZ',
 'BPI20_RequestForPayment_CZ']

In [6]:
dataset = "bpi_2012_CZ"

In [7]:
nan_methods = ["odd", "even", "random", "window","attr_level"]

masked_datasets = {key : pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_masked_{key}_all.csv") for key in nan_methods}


In [8]:
tab_all = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_all.csv") 
tab_all.head()

,CaseID,Activity,org:resource,time:timestamp,(case) AMOUNT_REQ,concept:name,lifecycle:transition
0,1,A_SUBMITTED-COMPLETE,Value 1,0.000000,9.903538,A_SUBMITTED,COMPLETE
1,1,A_PARTLYSUBMITTED-COMPLETE,Value 1,0.288182,9.903538,A_PARTLYSUBMITTED,COMPLETE
2,1,A_PREACCEPTED-COMPLETE,Value 1,3.995629,9.903538,A_PREACCEPTED,COMPLETE
3,1,W_Completeren aanvraag-SCHEDULE,Value 1,4.013297,9.903538,W_Completeren aanvraag,SCHEDULE
4,1,W_Completeren aanvraag-START,Value 2,10.583623,9.903538,W_Completeren aanvraag,START


In [9]:
tab_train = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_train.csv")
tab_valid = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_valid.csv")
tab_test = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_test.csv")

In [10]:
if dataset == "BPI_Challenge_2012_W_Complete":
    tab_all["org:resource"] = tab_all["org:resource"].astype(np.str_)
    tab_train["org:resource"] = tab_train["org:resource"].astype(np.str_)
    tab_valid["org:resource"] = tab_valid["org:resource"].astype(np.str_)
    tab_test["org:resource"] = tab_test["org:resource"].astype(np.str_)

In [11]:
with open("dataset_features.json", 'r') as file:
    dataset_info = json.load(file)[dataset]

In [12]:
dataset_info

{'categorical': ['Activity',
  'org:resource',
  'concept:name',
  'lifecycle:transition'],
 'numerical': ['time:timestamp', '(case) AMOUNT_REQ']}

In [13]:
if ACT_TIME_ONLY:
    categorical_columns = ["Activity"]
    real_value_columns = ["time:timestamp"]
    dataset = f"{dataset}_AT_only"
else:
    categorical_columns = dataset_info["categorical"]
    real_value_columns = dataset_info["numerical"]

In [14]:
for k in categorical_columns:
    tab_all[k] = tab_all[k].astype("object")
    tab_train[k] = tab_train[k].astype("object")
    tab_valid[k] = tab_valid[k].astype("object")
    tab_test[k] = tab_test[k].astype("object")
    
    for k_m in masked_datasets:
        masked_datasets[k_m][k] = masked_datasets[k_m][k].astype("object")

In [15]:
from numpy import NaN
if dataset == "sp2020_CZ":
    tab_all["REPAIR_IN_TIME_5D"] = [float(x) if x is not NaN else x for x in tab_all["REPAIR_IN_TIME_5D"].values]
    tab_train["REPAIR_IN_TIME_5D"] = [float(x) if x is not NaN else x for x in tab_train["REPAIR_IN_TIME_5D"].values]
    tab_valid["REPAIR_IN_TIME_5D"] = [float(x) if x is not NaN else x for x in tab_valid["REPAIR_IN_TIME_5D"].values]
    tab_test["REPAIR_IN_TIME_5D"] = [float(x) if x is not NaN else x for x in tab_test["REPAIR_IN_TIME_5D"].values]
    for k_m in masked_datasets:
        masked_datasets[k_m]["REPAIR_IN_TIME_5D"] = [float(x) if x is not NaN else x for x in masked_datasets[k_m]["REPAIR_IN_TIME_5D"].values]

In [16]:
for k in categorical_columns:
    print(f"{k} {tab_test[k].values.dtype}")

Activity object


In [17]:
dataset

'bpi_2012_CZ_AT_only'

In [18]:
from numpy import NaN
from math import log
if dataset == "BPI20_RequestForPayment_CZ":
    tab_all["case:RequestedAmount"] = [log(x) if x > 0 else 0. for x in tab_all["case:RequestedAmount"].values]
    tab_train["case:RequestedAmount"] = [log(x) if x > 0 else 0. for x in tab_train["case:RequestedAmount"].values]
    tab_valid["case:RequestedAmount"] = [log(x) if x > 0 else 0. for x in tab_valid["case:RequestedAmount"].values]
    tab_test["case:RequestedAmount"] = [log(x) if x > 0 else 0. for x in tab_test["case:RequestedAmount"].values]
    for k_m in masked_datasets:
        masked_datasets[k_m]["case:RequestedAmount"] = [log(x) if x > 0 else 0. if x == 0 else x for x in masked_datasets[k_m]["case:RequestedAmount"].values]

### Prepare the graphs

In [19]:
from torch import tensor,int64, float32
from torch_geometric.data import HeteroData


In [20]:
from pipeline import MISSING_VALUE

In [21]:
import sklearn.preprocessing
from pipeline import get_one_hot_encoder

In [22]:
from pipeline import add_new_timestamp

In [23]:
from pipeline import get_node_features

In [24]:
from pipeline import compute_edges_indexs

In [25]:
dataset

'bpi_2012_CZ_AT_only'

In [26]:
def get_masked_trace(dataset_traces, cat_features, real_features, caseid):
    trace = (
        dataset_traces.query(f"CaseID == '{caseid}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )
    
    
    
    if dataset != "bpi_2012_CZ" and dataset != 'bpi_2012_CZ_AT_only':
        mask = trace[trace.columns].isnull().apply(lambda x: all(x), axis=1)
    else:
        mask = trace[trace.columns].isnull().any(axis=1).values
        
    mask_index = [i for i in range(len(mask)) if mask[i]]
    
    for k in cat_features:
        for i in mask_index:
            trace.loc[i, k] = MISSING_VALUE
    
    for k in real_features:
        for i in mask_index:
            trace.loc[i, k] = -1
    
    return trace, mask
    
    

In [27]:
import torch 
from pipeline import get_masked_trace_dict_mask    

In [28]:
nan_methods[0]

'odd'

In [29]:
masked_datasets[nan_methods[0]]

,CaseID,Activity,org:resource,time:timestamp,(case) AMOUNT_REQ,concept:name,lifecycle:transition
0,1,A_SUBMITTED-COMPLETE,Value 1,0.000000,9.903538,A_SUBMITTED,COMPLETE
1,1,NaN,NaN,NaN,NaN,NaN,NaN
2,1,A_PREACCEPTED-COMPLETE,Value 1,3.995629,9.903538,A_PREACCEPTED,COMPLETE
3,1,NaN,NaN,NaN,NaN,NaN,NaN
4,1,W_Completeren aanvraag-START,Value 2,10.583623,9.903538,W_Completeren aanvraag,START
...,...,...,...,...,...,...,...
262195,13087,A_PARTLYSUBMITTED-COMPLETE,Value 1,16.390955,9.615872,A_PARTLYSUBMITTED,COMPLETE
262196,13087,W_Afhandelen leads-SCHEDULE,Value 1,16.390958,9.615872,W_Afhandelen leads,SCHEDULE
262197,13087,W_Afhandelen leads-START,Value 33,16.393580,9.615872,W_Afhandelen leads,START
262198,13087,A_DECLINED-COMPLETE,Value 33,16.393584,9.615872,A_DECLINED,COMPLETE


In [30]:

from copy import copy
import torch
from pipeline import build_prefixes_graph_from_trace

## Create the datasets

In [31]:
case_train_ids = get_case_ids(tab_train)
case_valid_ids = get_case_ids(tab_valid)
case_test_ids = get_case_ids(tab_test)

In [32]:
print(len(case_train_ids))
print(len(case_valid_ids))
print(len(case_test_ids))

7852
2617
2618


In [33]:
#trace2 = (
#        masked_datasets["odd"].query(f"CaseID == '{case_train_ids[0]}'")
#        .reset_index()
#        .drop(columns="index")
#        .drop(columns="CaseID")
#    )
#trace2

In [34]:
tab_train["CaseID"] = tab_train["CaseID"].astype(np.str_)
tab_valid["CaseID"] = tab_valid["CaseID"].astype(np.str_)
tab_test["CaseID"] = tab_test["CaseID"].astype(np.str_)

In [35]:
for k in masked_datasets:
    masked_datasets[k]["CaseID"] = masked_datasets[k]["CaseID"].astype(np.str_)

In [36]:
#tab_all["REPAIR_IN_TIME_5D"]

In [37]:
#masked_datasets["even"]["REPAIR_IN_TIME_5D"]

In [38]:
#trace = (
#        tab_train.query(f"CaseID == '{case_train_ids[1]}'")
#        .reset_index()
#        .drop(columns="index")
#        .drop(columns="CaseID")
#    )
#trace 

In [39]:
#trace3 = add_new_timestamp(trace)
#trace3

In [40]:
#m_trace, mask = get_masked_trace_dict_mask(masked_datasets["attr_level"], categorical_columns, real_value_columns, case_train_ids[1], trace3["time:timestamp"].values)

In [41]:
#m_trace

In [42]:
#mask

In [43]:
#graphs = build_prefixes_graph_from_trace(tab_all, trace, categorical_columns, real_value_columns, case_train_ids[1])

In [44]:
#graphs[0].x_dict["time:timestamp"]

In [45]:
#graphs[0].y

In [46]:
#graphs[0].masks

In [47]:
#graphs[0].y["Activity"][graphs[0].masks["Activity"]]

In [48]:
from tqdm.notebook import tqdm

In [49]:
from pipeline import build_split_graphs
import pickle

print("Preparing training dataset...")

X_train = build_split_graphs(
    tab_all, tab_train, categorical_columns, real_value_columns,
    masked_datasets, nan_methods, case_ids=case_train_ids
)
with open(data_dir_graphs + dataset + "_TRAIN_V2_repair.pkl", "wb") as f:
    pickle.dump(X_train, f)
del X_train
print("Done!\n\n")


Preparing training dataset...


  0%|          | 0/7852 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [56]:
print("Preparing validation dataset...")

X_valid = build_split_graphs(
    tab_all, tab_valid, categorical_columns, real_value_columns,
    masked_datasets, nan_methods, case_ids=case_valid_ids
)
with open(data_dir_graphs + dataset + "_VALID_V2_repair.pkl", "wb") as f:
    pickle.dump(X_valid, f)
del X_valid
print("Done!\n\n")

Preparing validation dataset...


  0%|          | 0/1377 [00:00<?, ?it/s]

Done!




In [57]:
print("Preparing test dataset...")

X_test = build_split_graphs(
    tab_all, tab_test, categorical_columns, real_value_columns,
    masked_datasets, nan_methods, case_ids=case_test_ids
)
with open(data_dir_graphs + dataset + "_TEST_V2_repair.pkl", "wb") as f:
    pickle.dump(X_test, f)
del X_test
print("Done!\n\n")

Preparing test dataset...


  0%|          | 0/1378 [00:00<?, ?it/s]

Done!




In [58]:
from pipeline import build_test_graphs_for_type

In [59]:
def create_and_save_test(case: str):
    X_test = build_test_graphs_for_type(
        tab_all, tab_test, categorical_columns, real_value_columns,
        masked_datasets, nan_methods, mask_type=case, case_ids=case_test_ids
    )
    print(data_dir_graphs + dataset + f"_TEST_V2_repair_{case}.pkl")
    with open(data_dir_graphs + dataset + f"_TEST_V2_repair_{case}.pkl", "wb") as f:
        pickle.dump(X_test, f)

In [60]:
data_dir_graphs

'/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/'

In [61]:
create_and_save_test("even")

  0%|          | 0/1378 [00:00<?, ?it/s]

/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/BPI20_RequestForPayment_CZ_AT_only_TEST_V2_repair_even.pkl


In [62]:
create_and_save_test("odd")

  0%|          | 0/1378 [00:00<?, ?it/s]

/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/BPI20_RequestForPayment_CZ_AT_only_TEST_V2_repair_odd.pkl


In [63]:
create_and_save_test("random")

  0%|          | 0/1378 [00:00<?, ?it/s]

/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/BPI20_RequestForPayment_CZ_AT_only_TEST_V2_repair_random.pkl


In [64]:
create_and_save_test("window")

  0%|          | 0/1378 [00:00<?, ?it/s]

/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/BPI20_RequestForPayment_CZ_AT_only_TEST_V2_repair_window.pkl


In [65]:
create_and_save_test("attr_level")

  0%|          | 0/1378 [00:00<?, ?it/s]

/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/BPI20_RequestForPayment_CZ_AT_only_TEST_V2_repair_attr_level.pkl
